# MangoLeaf AI — Rejection Threshold Fitting (v3)

**This notebook does not train anything.** It loads the finished Trial 3 model out of the
resume bundle, measures how it behaves on out-of-scope photographs, and fills in the two
rejection thresholds that the training run left unusable.

No weight is changed. Test accuracy stays 97.46%. The only file this produces is a
replacement `public/model/model_config.json`.

---

## Why this notebook exists

The training run fitted `marginThreshold: 0.00`. A margin of zero is always true, so that
check rejects nothing and the app is running on two of its three gates instead of three.

The cause is a tie-break, not a bad model. The grid search kept a candidate only when its
rejection rate was strictly better than the best so far, so among many `(margin, entropy)`
pairs that all achieved the same best rejection rate, the first one tried won — and the
sweep starts at `margin = 0.00`. Trial 2.1 hit the same thing: 56 pairs tied at the
optimum, and a margin of 0.55 was available at exactly the same rejection rate and zero
cost in retention.

Cell 4 below fixes the tie-break: among pairs that tie on rejection, it keeps the
**largest** margin. Cell 5 then audits the result and prints the whole tie set so the
choice is on the record rather than implicit.

## What it needs in Google Drive

```
MyDrive/
  mangoleaf_v3/resume/
    mangoleaf_v3_calibrated.keras     34.70 MB
    y_prob_cal.npy
    y_true.npy
    resume_info.json
  mango_ood_test/
    tune/                              84 non-mango photographs
    report/                            84 non-mango photographs
```

Both already exist. `resume/` was written by the last cell of the Trial 3 training run.
The resume bundle holds the test-set probabilities but **not** the out-of-scope ones, so
the 168 non-mango images are run through the loaded model again here. That is inference
only: `model.predict`, never `model.fit`.

## How to run

Runtime, then Run all. Eight code cells, in order:

| Cell | What it does |
| :-- | :-- |
| 1 | mounts Drive, loads the finished model and the saved test predictions, asserts they are the Trial 3 pair |
| 2 | splits the 168 out-of-scope images into the 84 / 84 tune and report halves |
| 3 | runs `model.predict` on both halves, draws the three histograms |
| 4 | sweeps margin and entropy with the tie-break fixed, prints the fitted thresholds |
| 5 | audits the fitted pair: the full tie set, and how close images sit to the boundaries |
| 6 | measures the real float16 cost, skipping the one tensor that cannot fit |
| 7 | scales the simulated bands up to the measured export shift and re-stresses the pair |
| 8 | writes the new `model_config.json` and prints it for copy-paste |

Confidence stays pinned at **0.70**. It is a manuscript claim cited to [31] and no cell in
this notebook touches it. Only margin and entropy are fitted.

**Do not** run `MangoLeaf_AI_Training_v3_Post_Training.ipynb` again. It holds the figures
and tables the manuscript cites, and a rerun would overwrite them with different numbers.
**Do not** retrain to fit thresholds. New weights would not match the ones already
measured at 97.46%.

In [ ]:
# ============================================================================
# CELL 1 - Resume the finished Trial 3 run. Loads, asserts, trains nothing.
# ============================================================================
from google.colab import drive
drive.mount("/content/drive")

import os, json, random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

OOD_DIR    = "/content/drive/MyDrive/mango_ood_test"
DRIVE_SAVE = "/content/drive/MyDrive/mangoleaf_v3"
RESUME_DIR = os.path.join(DRIVE_SAVE, "resume")

IMG_SIZE = (224, 224)
IMG_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

CONFIDENCE_THRESHOLD = 0.70      # FIXED by the manuscript. No cell here tunes it.
MIN_ID_RETENTION     = 0.95

CLASS_NAMES = ["anthracnose", "bacterial_black_spot", "bacterial_canker",
               "healthy", "mango_scab", "sooty_mold"]

EXPECTED_TRIAL    = "3"
EXPECTED_TEST_ACC = 0.974555     # Trial 3, post-training cell #17

FIG_DIR = "/content/figures"
FIG_DPI = 150
os.makedirs(FIG_DIR, exist_ok=True)

# ---- 1. the resume bundle must be complete ---------------------------------
NEEDED = ["mangoleaf_v3_calibrated.keras", "y_prob_cal.npy",
          "y_true.npy", "resume_info.json"]
missing = [n for n in NEEDED if not os.path.exists(os.path.join(RESUME_DIR, n))]
if missing:
    raise FileNotFoundError(
        "Missing in {}: {}\n"
        "These are written by the last cell of the Trial 3 training notebook. "
        "Do not retrain to recreate them: new weights would not match the ones "
        "already measured at {:.2%}.".format(RESUME_DIR, missing, EXPECTED_TEST_ACC))

with open(os.path.join(RESUME_DIR, "resume_info.json")) as f:
    resume_info = json.load(f)

model      = tf.keras.models.load_model(
    os.path.join(RESUME_DIR, "mangoleaf_v3_calibrated.keras"), compile=False)
y_prob_cal = np.load(os.path.join(RESUME_DIR, "y_prob_cal.npy"))
y_true     = np.load(os.path.join(RESUME_DIR, "y_true.npy"))

# ---- 2. it must be the Trial 3 bundle, not Trial 2.1's ---------------------
assert y_prob_cal.shape[1] == len(CLASS_NAMES), \
    "probabilities have {} columns, expected {}".format(
        y_prob_cal.shape[1], len(CLASS_NAMES))
assert len(y_prob_cal) == len(y_true), "probabilities and labels differ in length"
assert list(resume_info["classes"]) == CLASS_NAMES, \
    "class order differs from the app. Stop: the config would mislabel every scan."
assert resume_info["temperatureFoldedIntoWeights"] is True, \
    "temperature is not folded into these weights"
assert str(resume_info["trial"]) == EXPECTED_TRIAL, \
    "this is the Trial {} bundle, not Trial {}".format(
        resume_info["trial"], EXPECTED_TRIAL)
assert abs(float(resume_info["testAccuracy"]) - EXPECTED_TEST_ACC) < 1e-4, \
    "bundle records {:.6f} test accuracy, Trial 3 measured {:.6f}".format(
        float(resume_info["testAccuracy"]), EXPECTED_TEST_ACC)

# The saved probabilities must still reproduce the accuracy they were saved with.
# This checks the two .npy files belong together and to this run. It does not
# re-verify the model: the 393 test images are not in the resume bundle, and
# rebuilding the split to get them is unnecessary for fitting thresholds.
reloaded_acc = float(accuracy_score(y_true, y_prob_cal.argmax(axis=1)))
assert abs(reloaded_acc - float(resume_info["testAccuracy"])) < 1e-4, \
    "accuracy drift - wrong resume files"

T = float(resume_info["temperature"])
assert abs(resume_info["confidenceThreshold"] - CONFIDENCE_THRESHOLD) < 1e-9, \
    "the bundle was fitted against a different confidence threshold"

print("=" * 70)
print("RESUMED WITHOUT RETRAINING")
print("=" * 70)
print("  trial                    : {}".format(resume_info["trial"]))
print("  model                    : mangoleaf_v3_calibrated.keras")
print("  temperature folded in    : T = {:.6f}".format(T))
print("  test images with saved probabilities : {}".format(len(y_true)))
print("  test accuracy recomputed : {:.6f}  ({:.2%})".format(reloaded_acc, reloaded_acc))
print("  confidence threshold     : {:.2f}  (fixed, not tuned here)".format(
    CONFIDENCE_THRESHOLD))
print("  retention floor          : {:.0%}".format(MIN_ID_RETENTION))
print("")
print("  Nothing below calls model.fit. Weights are read-only from here on.")

# ---- 3. the out-of-scope folder must be there ------------------------------
if not os.path.isdir(OOD_DIR):
    raise FileNotFoundError(
        "{} not found. It holds the non-mango photographs the thresholds are "
        "measured against.".format(OOD_DIR))
print("")
print("Out-of-scope folder: {}".format(OOD_DIR))
for entry in sorted(os.listdir(OOD_DIR)):
    p = os.path.join(OOD_DIR, entry)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.lower().endswith(IMG_EXTS)])
        print("  {:<12} {} images".format(entry + "/", n))

## Cells 2 to 4 — Fitting the two thresholds

This measures the behaviour reported from live use: non-mango photographs coming back with a
disease label and a confident-looking score.

### The three signals

| Signal | What it catches |
| :-- | :-- |
| **confidence** — highest probability | model is unsure overall |
| **margin** — top probability minus second | two classes nearly tied, e.g. 0.71 against 0.28 |
| **entropy** — how evenly probability spreads | probability smeared thinly across several classes |

Confidence alone misses the last two. A prediction of 0.71 / 0.28 / 0.01 clears the 70% bar
while being a coin flip between two diseases.

### What gets tuned, and what does not

**Confidence stays fixed at 0.70.** It is a manuscript claim, cited to [31], and this
notebook never touches it. Only margin and entropy are fitted.

### The data

`mango_ood_test/` holds non-mango photographs. They never train anything. `model.predict` is
called on them, never `model.fit`. There are no labels, no gradients, no weight change. They
never enter the confusion matrix and never affect the 97.46% test accuracy. They are a
measuring stick, opened once, after the model is finished.

Thresholds are fitted on `tune/` and reported on `report/`. Fitting and reporting on the same
images would be grading yourself on the answers you studied — the same reason the mango data
has a separate test split. Trial 2.1 saw the two halves agree; Trial 3 may not, and if they
disagree the **report half is the number that goes in the manuscript**.

### The tie-break, stated plainly

Many `(margin, entropy)` pairs reach the same best rejection rate. Trial 3's sweep, as
written in the training notebook, kept whichever it tried first, and it tries `margin = 0.00`
first. Cell 4 keeps the largest margin instead. Same rejection rate, same retention, but the
margin gate now actually rejects something — which is the whole point of having it.

In [ ]:
# ============================================================================
# CELL 2 - Helpers, and the 84 / 84 out-of-scope split.
# ============================================================================

def prob_stats(p):
    """Returns (top1 confidence, margin, normalized entropy) for each row."""
    top1 = p.max(axis=1)
    srt  = np.sort(p, axis=1)
    margin = srt[:, -1] - srt[:, -2]
    ent = -(p * np.log(p + 1e-12)).sum(axis=1) / np.log(p.shape[1])
    return top1, margin, ent


def load_flat(paths):
    """Loads images as raw 0-255 float32. MobileNetV3 rescales internally."""
    out = []
    for p in paths:
        try:
            im = Image.open(p).convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
            out.append(np.array(im, dtype=np.float32))
        except Exception as e:
            print("  skipped", os.path.basename(p), repr(e))
    return np.stack(out) if out else None


def list_images(folder):
    return sorted(os.path.join(folder, f) for f in os.listdir(folder)
                  if f.lower().endswith(IMG_EXTS))


tune_dir = os.path.join(OOD_DIR, "tune")
rep_dir  = os.path.join(OOD_DIR, "report")

if os.path.isdir(tune_dir) and os.path.isdir(rep_dir):
    tune_p = list_images(tune_dir)
    rep_p  = list_images(rep_dir)
    print("tune/ and report/ subfolders found. Using them as they are.")
else:
    # Flat folder: split once with a fixed seed so the halves never move
    # between sessions.
    allp = list_images(OOD_DIR)
    if not allp:
        raise FileNotFoundError("No images found in {}".format(OOD_DIR))
    perm = np.random.RandomState(SEED).permutation(len(allp))
    half = len(allp) // 2
    tune_p = [allp[i] for i in sorted(perm[:half])]
    rep_p  = [allp[i] for i in sorted(perm[half:])]
    print("Flat folder found. Split 50/50 with seed {}.".format(SEED))

print("Out-of-scope images  tune: {}   report: {}".format(len(tune_p), len(rep_p)))

if min(len(tune_p), len(rep_p)) < 20:
    print("")
    print("WARNING: fewer than 20 images on a side. The rejection rate this")
    print("produces has a confidence interval too wide to quote. Gather more")
    print("non-mango photographs before treating the number as a result.")

In [ ]:
# ============================================================================
# CELL 3 - Run the out-of-scope images through the model. Inference only.
# ============================================================================
X_tune = load_flat(tune_p)
X_rep  = load_flat(rep_p)
if X_tune is None or X_rep is None:
    raise RuntimeError("One half loaded zero images. Check the file extensions.")

p_tune = model.predict(X_tune, verbose=0)
p_rep  = model.predict(X_rep,  verbose=0)

id_top1, id_marg, id_ent = prob_stats(y_prob_cal)   # in-distribution: mango test set
ot_top1, ot_marg, ot_ent = prob_stats(p_tune)       # out-of-scope, tune half
or_top1, or_marg, or_ent = prob_stats(p_rep)        # out-of-scope, report half

print("=" * 70)
print("SIGNAL SEPARATION")
print("=" * 70)
print("{:<24} {:>14} {:>14} {:>14}".format(
    "", "mango test", "non-mango tune", "non-mango rep"))
rows = [("mean confidence", id_top1, ot_top1, or_top1),
        ("mean margin",     id_marg, ot_marg, or_marg),
        ("mean entropy",    id_ent,  ot_ent,  or_ent)]
for label, a, b, c in rows:
    print("{:<24} {:>14.4f} {:>14.4f} {:>14.4f}".format(
        label, a.mean(), b.mean(), c.mean()))
print("{:<24} {:>13.1f}% {:>13.1f}% {:>13.1f}%".format(
    "already below {:.2f} conf".format(CONFIDENCE_THRESHOLD),
    100 * (id_top1 < CONFIDENCE_THRESHOLD).mean(),
    100 * (ot_top1 < CONFIDENCE_THRESHOLD).mean(),
    100 * (or_top1 < CONFIDENCE_THRESHOLD).mean()))
print("")
print("The wider the gap in each row, the more work that signal can do. A row")
print("with similar numbers on both sides cannot separate mango from non-mango")
print("no matter where the threshold is put.")

# ---- three histograms ------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
panels = [("Top-1 confidence", id_top1, np.concatenate([ot_top1, or_top1]),
           CONFIDENCE_THRESHOLD),
          ("Margin (top minus second)", id_marg,
           np.concatenate([ot_marg, or_marg]), None),
          ("Normalized entropy", id_ent, np.concatenate([ot_ent, or_ent]), None)]
for ax, (title, a, b, line) in zip(axes, panels):
    bins = np.linspace(0, 1, 41)
    ax.hist(a, bins=bins, alpha=0.65, label="mango test ({})".format(len(a)),
            color="#277839")
    ax.hist(b, bins=bins, alpha=0.65, label="non-mango ({})".format(len(b)),
            color="#c0392b")
    if line is not None:
        ax.axvline(line, color="black", linestyle="--", linewidth=1.2,
                   label="fixed {:.2f}".format(line))
    ax.set_title(title)
    ax.set_xlabel("value")
    ax.set_ylabel("images")
    ax.legend(fontsize=8)
plt.suptitle("Rejection signals, Trial 3", y=1.02)
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "rejection_distributions_refit.png")
plt.savefig(fig_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
print("")
print("Saved {}".format(fig_path))

In [ ]:
# ============================================================================
# CELL 4 - Fit margin and entropy. Confidence stays at 0.70 throughout.
#
# The tie-break is the fix. Many (margin, entropy) pairs reach the same best
# rejection rate. The training notebook kept whichever it tried first, and the
# sweep starts at margin 0.00, which is always true - so the margin gate ended
# up rejecting nothing. Here, pairs that tie on rejection are broken by taking
# the LARGEST margin, and entropy descends from 1.00 so the loosest entropy
# (highest retention) wins within that margin.
# ============================================================================
GRID_M = np.round(np.arange(0.00, 0.96, 0.01), 2)   # snapped to 2dp so the
GRID_E = np.round(np.arange(1.00, 0.04, -0.01), 2)  # printed value IS the
EPS    = 1e-12                                      # value actually used


def score(m, e):
    keep_id  = ((id_top1 >= CONFIDENCE_THRESHOLD) &
                (id_marg >= m) & (id_ent <= e))
    keep_ood = ((ot_top1 >= CONFIDENCE_THRESHOLD) &
                (ot_marg >= m) & (ot_ent <= e))
    return float(keep_id.mean()), float(1.0 - keep_ood.mean()), keep_id


best     = None    # fixed tie-break: largest margin among ties
best_old = None    # the training notebook's rule, kept only to show the difference
n_feasible = 0

for m in GRID_M:
    for e in GRID_E:
        retention, rejection, _ = score(m, e)
        if retention < MIN_ID_RETENTION:
            continue
        n_feasible += 1
        cand = {"margin": float(m), "entropy": float(e),
                "retention": retention, "rejection": rejection}

        # what the training notebook did: strict >, so first-tried wins ties
        if best_old is None or rejection > best_old["rejection"]:
            best_old = dict(cand)

        # fixed: on a tie, prefer the larger margin
        if best is None:
            best = dict(cand)
        elif rejection > best["rejection"] + EPS:
            best = dict(cand)
        elif abs(rejection - best["rejection"]) <= EPS and m > best["margin"] + EPS:
            best = dict(cand)

if best is None:
    raise RuntimeError(
        "No (margin, entropy) pair keeps {:.0%} of the mango test set. Either the "
        "retention floor is too strict or the model is not separating at all."
        .format(MIN_ID_RETENTION))

MARGIN_THRESHOLD  = round(best["margin"], 2)
ENTROPY_THRESHOLD = round(best["entropy"], 2)

# ---- evaluate the chosen pair on the held-out report half -------------------
keep_id  = ((id_top1 >= CONFIDENCE_THRESHOLD) &
            (id_marg >= MARGIN_THRESHOLD) & (id_ent <= ENTROPY_THRESHOLD))
keep_rep = ((or_top1 >= CONFIDENCE_THRESHOLD) &
            (or_marg >= MARGIN_THRESHOLD) & (or_ent <= ENTROPY_THRESHOLD))
report_rejection = float(1.0 - keep_rep.mean())
acc_on_kept = float(accuracy_score(y_true[keep_id],
                                   y_prob_cal.argmax(axis=1)[keep_id]))

print("=" * 70)
print("RECOMMENDED THRESHOLDS")
print("=" * 70)
print("  confidence >= {:.2f}     (FIXED by manuscript, not tuned)".format(
    CONFIDENCE_THRESHOLD))
print("  margin     >= {:.2f}".format(MARGIN_THRESHOLD))
print("  entropy    <= {:.2f}".format(ENTROPY_THRESHOLD))
print("")
print("Measured on the tune half:")
print("  in-distribution retention : {:.1%}  ({} of {} test images kept)".format(
    best["retention"], int(keep_id.sum()), len(y_true)))
print("  out-of-scope rejection    : {:.1%}  ({} of {})".format(
    best["rejection"], round(best["rejection"] * len(ot_ent)), len(ot_ent)))
print("")
print("Measured on the held-out report half (THIS is the manuscript number):")
print("  out-of-scope rejection    : {:.1%}  ({} of {})".format(
    report_rejection, round(report_rejection * len(or_ent)), len(or_ent)))
se = (report_rejection * (1 - report_rejection) / len(or_ent)) ** 0.5
print("  95% CI                    : {:.1%} to {:.1%}".format(
    max(0.0, report_rejection - 1.96 * se),
    min(1.0, report_rejection + 1.96 * se)))
print("")
print("Accuracy on the mango predictions that pass all three checks: {:.2%}".format(
    acc_on_kept))

print("")
print("=" * 70)
print("WHAT THE TIE-BREAK CHANGED")
print("=" * 70)
print("  feasible (margin, entropy) pairs at the {:.0%} floor : {}".format(
    MIN_ID_RETENTION, n_feasible))
print("  training notebook's rule would have chosen : margin {:.2f}  entropy {:.2f}".format(
    best_old["margin"], best_old["entropy"]))
print("  this cell chooses                          : margin {:.2f}  entropy {:.2f}".format(
    MARGIN_THRESHOLD, ENTROPY_THRESHOLD))
if abs(best_old["rejection"] - best["rejection"]) > EPS:
    print("  Rejection rates differ. Stop and read cell 5 before deploying.")
elif abs(best_old["margin"] - MARGIN_THRESHOLD) < EPS:
    print("  Same pair. The optimum was not tied, so nothing needed fixing.")
else:
    print("  Identical rejection rate ({:.1%} on tune), so the larger margin is".format(
        best["rejection"]))
    print("  free. The margin gate now rejects something instead of nothing.")

reject_cfg = {
    "confidenceThreshold": CONFIDENCE_THRESHOLD,
    "marginThreshold":     MARGIN_THRESHOLD,
    "entropyThreshold":    ENTROPY_THRESHOLD,
    "tuneRetention":       round(best["retention"], 4),
    "tuneRejection":       round(best["rejection"], 4),
    "reportRejection":     round(report_rejection, 4),
    "accuracyOnAccepted":  round(acc_on_kept, 4),
    "oodTuneCount":        int(len(ot_ent)),
    "oodReportCount":      int(len(or_ent)),
}

## Cells 5 to 7 — Audit before deploying

Cell 4 produced a pair of numbers. These three cells check that the pair survives contact with
the phone before it goes anywhere near `public/model/`.

**Cell 5** reads only. It reports raw counts instead of percentages, prints the whole tie set
so the margin choice is documented rather than implicit, and counts how many images sit close
enough to a boundary that a small numerical shift could flip them.

**Cell 6** measures that shift. The web export stores weights and runs activations at 16-bit
precision, so the phone's probabilities differ slightly from this notebook's. One BatchNorm
variance tensor in this model exceeds `float16`'s maximum of 65504, and casting it produces
`inf`, which silently breaks that channel. The training notebook's cell 32 hit exactly that and
its `RuntimeWarning: overflow encountered in cast` is why its shift figure was measured on a
damaged model. Cell 6 leaves the overflowing tensors at `float32` and quantizes the rest. That
is faithful to what actually ships, because `tensorflowjs_converter` folds BatchNorm statistics
into the preceding convolution — they do not exist as separate tensors in the deployed graph.

**Cell 7** scales the simulated band up to the shift measured on the real export and re-stresses
the chosen pair. The simulation covers weight quantization only; the phone also runs
`WEBGL_FORCE_F16_TEXTURES`, so activations are 16-bit too and the real shift is larger.

### What to do with the result

If retention holds above 95% and rejection holds up under the stressed band, deploy the pair.
If retention drops below 95% in the worst direction, the entropy threshold needs loosening by
hand before deploying — not the confidence threshold, which stays at 0.70 regardless.

Cell 8 refuses to write the config at all if the margin lands below 0.05, because a margin that
small is inside the float16 band and the phone could accept what this notebook rejected.

In [ ]:
# ============================================================================
# CELL 5 - Audit the fitted thresholds. Fits nothing new, reads only.
# Answers three questions:
#   1. How few images are these percentages actually built on?
#   2. How many (margin, entropy) pairs tie at this rejection rate, and what is
#      the full range of margins available for free?
#   3. How many images sit close enough to a boundary that float16 export could
#      flip them on the phone?
# ============================================================================
FLOAT16_SHIFT = 0.042463   # measured in the training notebook, cell #32


def evaluate(m, e):
    kid = (id_top1 >= CONFIDENCE_THRESHOLD) & (id_marg >= m) & (id_ent <= e)
    kot = (ot_top1 >= CONFIDENCE_THRESHOLD) & (ot_marg >= m) & (ot_ent <= e)
    kor = (or_top1 >= CONFIDENCE_THRESHOLD) & (or_marg >= m) & (or_ent <= e)
    return float(kid.mean()), float(1 - kot.mean()), float(1 - kor.mean()), kid


n_id, n_t, n_r = len(id_ent), len(ot_ent), len(or_ent)

print("=" * 70)
print("1. RAW COUNTS (percentages hide how few images these are)")
print("=" * 70)
ret0, rej_t0, rej_r0, kid0 = evaluate(MARGIN_THRESHOLD, ENTROPY_THRESHOLD)
print("  test images accepted        : {} of {}".format(int(kid0.sum()), n_id))
print("  non-mango rejected, tune    : {} of {}".format(round(rej_t0 * n_t), n_t))
print("  non-mango rejected, report  : {} of {}".format(round(rej_r0 * n_r), n_r))
se = (rej_r0 * (1 - rej_r0) / n_r) ** 0.5
print("  95% CI on the report number : {:.1%} to {:.1%}".format(
    max(0.0, rej_r0 - 1.96 * se), min(1.0, rej_r0 + 1.96 * se)))
print("  Quote it as '{:.1%} ({} of {})', not bare {:.1%}.".format(
    rej_r0, round(rej_r0 * n_r), n_r, rej_r0))
gap = abs(rej_t0 - rej_r0)
print("")
print("  tune half {:.1%} against report half {:.1%}: a {:.1f} point gap.".format(
    rej_t0, rej_r0, 100 * gap))
if gap > 0.10:
    print("  That gap is large. The tune half is where the thresholds were fitted,")
    print("  so it is optimistic by construction. Quote the report half only.")

print("")
print("=" * 70)
print("2. THE TIE SET - HOW MUCH MARGIN WAS AVAILABLE FOR FREE")
print("=" * 70)
print("Searching every (margin, entropy) pair that holds retention at or above")
print("{:.0%} and rejection at exactly {:.4f} on the tune half.".format(
    MIN_ID_RETENTION, rej_t0))

ties = []
for m in GRID_M:
    for e in GRID_E:
        ret, rej_t, _, _ = evaluate(m, e)
        if ret >= MIN_ID_RETENTION and abs(rej_t - rej_t0) < 1e-9:
            ties.append((float(m), float(e), ret))

if not ties:
    print("  No tie set found. The chosen pair is the unique optimum.")
else:
    margins = sorted({round(t[0], 4) for t in ties})
    print("")
    print("  equally-optimal (margin, entropy) pairs : {}".format(len(ties)))
    print("  distinct margins among them            : {} (from {:.2f} to {:.2f})".format(
        len(margins), margins[0], margins[-1]))
    print("  chosen                                 : {:.2f}".format(MARGIN_THRESHOLD))
    if abs(margins[-1] - MARGIN_THRESHOLD) > 1e-9:
        print("  MISMATCH: {:.2f} was available and was not taken. Read cell 4.".format(
            margins[-1]))
    elif MARGIN_THRESHOLD <= 0.0:
        print("  Still zero even after the tie-break. Entropy fully subsumes margin")
        print("  on this data: every image the margin gate would reject, the entropy")
        print("  gate already rejects. Say so in Limitations rather than shipping a")
        print("  threshold that does nothing.")
    else:
        print("  Largest available margin was taken. The margin gate does real work")
        print("  at no cost in rejection or retention.")

    # What each distinct margin costs, so the choice is on the record.
    print("")
    print("  {:<10} {:<10} {:<12} {:<12}".format(
        "margin", "entropy", "retention", "report rej"))
    shown = 0
    for mm in margins:
        cands = [t for t in ties if abs(t[0] - mm) < 1e-9]
        _, ee, _ = max(cands, key=lambda t: t[2])
        r, _, rr, _ = evaluate(mm, ee)
        mark = "  <- chosen" if abs(mm - MARGIN_THRESHOLD) < 1e-9 else ""
        print("  {:<10.2f} {:<10.2f} {:<12.1%} {:<12.1%}{}".format(
            mm, ee, r, rr, mark))
        shown += 1
        if shown >= 20 and len(margins) > 22:
            print("  ... {} more margins omitted".format(len(margins) - shown))
            break

print("")
print("=" * 70)
print("3. FLOAT16 EXPOSURE ON THE BOUNDARIES")
print("=" * 70)
print("Weight storage in float16 shifts confidence by up to {:.6f}, which".format(
    FLOAT16_SHIFT))
print("shifts margin and entropy too. Any image sitting within that distance of")
print("a threshold can decide differently on the phone than it does here.")
print("")
for label, arr, thr, n, sense in [
        ("mango test",       id_ent, ENTROPY_THRESHOLD, n_id, "entropy"),
        ("non-mango tune",   ot_ent, ENTROPY_THRESHOLD, n_t,  "entropy"),
        ("non-mango report", or_ent, ENTROPY_THRESHOLD, n_r,  "entropy"),
        ("mango test",       id_marg, MARGIN_THRESHOLD, n_id, "margin"),
        ("non-mango tune",   ot_marg, MARGIN_THRESHOLD, n_t,  "margin"),
        ("non-mango report", or_marg, MARGIN_THRESHOLD, n_r,  "margin")]:
    near = int((np.abs(arr - thr) <= FLOAT16_SHIFT).sum())
    print("  {:<18} {:<8} within +/-{:.3f} of {:.2f} : {} of {}  ({:.1f}%)".format(
        label, sense, FLOAT16_SHIFT, thr, near, n, 100 * near / n))

below = id_ent[id_ent <= ENTROPY_THRESHOLD]
above = or_ent[or_ent > ENTROPY_THRESHOLD]
print("")
if len(below):
    print("  nearest accepted mango entropy below the line : {:.4f}  (gap {:.4f})".format(
        below.max(), ENTROPY_THRESHOLD - below.max()))
if len(above):
    print("  nearest rejected non-mango entropy above it   : {:.4f}  (gap {:.4f})".format(
        above.min(), above.min() - ENTROPY_THRESHOLD))

print("")
print("  Worst case if every phone entropy rises by {:.6f}:".format(FLOAT16_SHIFT))
r_lo, _, rr_lo, _ = evaluate(MARGIN_THRESHOLD, ENTROPY_THRESHOLD - FLOAT16_SHIFT)
print("    retention {:.1%}   report rejection {:.1%}".format(r_lo, rr_lo))
print("  Worst case if every phone entropy falls by the same amount:")
r_hi, _, rr_hi, _ = evaluate(MARGIN_THRESHOLD, ENTROPY_THRESHOLD + FLOAT16_SHIFT)
print("    retention {:.1%}   report rejection {:.1%}".format(r_hi, rr_hi))
print("")
if r_lo >= MIN_ID_RETENTION:
    print("  Retention holds above {:.0%} in the worst direction. The entropy".format(
        MIN_ID_RETENTION))
    print("  threshold of {:.2f} is safe to deploy.".format(ENTROPY_THRESHOLD))
else:
    print("  WARNING: retention drops to {:.1%} in the worst direction, below the".format(
        r_lo))
    print("  {:.0%} floor. Loosen the entropy threshold by hand before deploying.".format(
        MIN_ID_RETENTION))

In [ ]:
# ============================================================================
# CELL 6 - Measure the real float16 cost, with the overflow handled.
#
# Casting every weight to float16 breaks this model: at least one BatchNorm
# variance exceeds float16's maximum of 65504, becomes inf, and that channel
# then emits only its beta. The training notebook's cell #32 did exactly that
# and printed "RuntimeWarning: overflow encountered in cast", so its shift
# figure was measured on a damaged model.
#
# Fix: leave the tensors that cannot fit at float32 and quantize the rest. This
# is faithful to what ships, because tensorflowjs_converter FOLDS BatchNorm
# statistics into the preceding convolution - they are not separate tensors in
# the deployed graph.
# ============================================================================
F16_MAX = float(np.finfo(np.float16).max)   # 65504.0

w32 = model.get_weights()
try:
    names = [v.path for v in model.weights]
except AttributeError:
    names = [v.name for v in model.weights]

over = [(i, names[i], float(np.abs(w).max()))
        for i, w in enumerate(w32) if np.abs(w).max() > F16_MAX]

print("=" * 70)
print("A. WHICH TENSORS OVERFLOW float16")
print("=" * 70)
print("float16 max representable: {:.0f}".format(F16_MAX))
print("tensors over that limit  : {} of {}".format(len(over), len(w32)))
for i, nm, mx in over[:12]:
    print("  [{:>3}] {:<52} max |w| = {:.1f}".format(i, nm[-52:], mx))
if len(over) > 12:
    print("  ... and {} more".format(len(over) - 12))
print("")
print("These stay float32 below. If they are all BatchNorm statistics the")
print("simulation is faithful, because the converter folds them into conv")
print("weights that ARE quantized. If any is a conv kernel or a dense weight,")
print("stop: the simulation would then be unsound and the numbers below void.")

skip = {i for i, _, _ in over}

# ---- quantize everything that can actually be represented -------------------
try:
    model.set_weights([w if i in skip else w.astype(np.float16).astype(np.float32)
                       for i, w in enumerate(w32)])
    p_tune16 = model.predict(X_tune, verbose=0)
    p_rep16  = model.predict(X_rep,  verbose=0)
finally:
    model.set_weights(w32)

assert np.abs(model.predict(X_tune, verbose=0) - p_tune).max() < 1e-6, \
    "float32 weights were not restored"

print("")
print("=" * 70)
print("B. IS THE SIMULATION CREDIBLE?")
print("=" * 70)
p16_all = np.concatenate([p_tune16, p_rep16])
p32_all = np.concatenate([p_tune,   p_rep])
finite  = bool(np.isfinite(p16_all).all())
agree   = float((p16_all.argmax(1) == p32_all.argmax(1)).mean())
print("  all probabilities finite (no inf/nan) : {}".format(finite))
print("  argmax agreement with float32         : {:.1%}".format(agree))
if not finite or agree < 0.95:
    raise RuntimeError(
        "Simulation still unsound. Do NOT use its numbers. Fall back to the "
        "{:.6f} confidence bound measured on the real export.".format(FLOAT16_SHIFT))
print("  Credible. Proceeding.")

t16 = prob_stats(p_tune16); r16 = prob_stats(p_rep16)
d_conf = np.concatenate([np.abs(t16[0] - ot_top1), np.abs(r16[0] - or_top1)])
d_marg = np.concatenate([np.abs(t16[1] - ot_marg), np.abs(r16[1] - or_marg)])
d_ent  = np.concatenate([np.abs(t16[2] - ot_ent),  np.abs(r16[2] - or_ent)])

print("")
print("=" * 70)
print("C. MEASURED float16 SHIFT, {} out-of-scope images".format(len(d_conf)))
print("=" * 70)
print("{:<12} {:>10} {:>10} {:>12}".format("signal", "mean", "p99", "max"))
for nm, d in [("confidence", d_conf), ("margin", d_marg), ("entropy", d_ent)]:
    print("{:<12} {:>10.6f} {:>10.6f} {:>12.6f}".format(
        nm, d.mean(), np.percentile(d, 99), d.max()))
print("")
print("  real export, confidence, cell #32 : {:.6f}".format(FLOAT16_SHIFT))
print("  this simulation, confidence, max  : {:.6f}".format(d_conf.max()))
print("  If these two are close, the entropy and margin maxima above are")
print("  trustworthy. If the simulation is far smaller, cell 7 scales it up.")

BAND_E = float(d_ent.max())
BAND_M = float(d_marg.max())


# ---- refit under the band, with vacuous thresholds banned -------------------
def worst_ret(m, e):
    """Retention if the phone moves every image the wrong way."""
    k = ((id_top1 >= CONFIDENCE_THRESHOLD) &
         (id_marg >= m + BAND_M) & (id_ent <= e - BAND_E))
    return float(k.mean())


def worst_rej(m, e, top1, marg, ent):
    """Rejection if the phone moves every image the wrong way."""
    k = ((top1 >= CONFIDENCE_THRESHOLD) &
         (marg >= max(0.0, m - BAND_M)) & (ent <= e + BAND_E))
    return float(1.0 - k.mean())


print("")
print("=" * 70)
print("D. REFIT UNDER THE BAND - margin above 0.00, entropy below 1.00")
print("=" * 70)

band_best = None
for floor in (0.95, 0.93, 0.90):
    band_best = None
    for m in np.round(np.arange(0.01, 0.96, 0.01), 2):    # 0.00 banned: always true
        for e in np.round(np.arange(0.99, 0.04, -0.01), 2):  # 1.00 banned: always true
            if worst_ret(m, e) < floor:
                continue
            rj = worst_rej(m, e, ot_top1, ot_marg, ot_ent)
            if band_best is None or rj > band_best[3] + 1e-12 or (
                    abs(rj - band_best[3]) < 1e-12 and m > band_best[0]):
                band_best = (float(m), float(e), worst_ret(m, e), rj)
    if band_best is None:
        print("  retention floor {:.0%}: no non-vacuous pair survives.".format(floor))
        continue
    M2, E2, RW, RJW = band_best
    kn = ((id_top1 >= CONFIDENCE_THRESHOLD) & (id_marg >= M2) & (id_ent <= E2))
    kr = ((or_top1 >= CONFIDENCE_THRESHOLD) & (or_marg >= M2) & (or_ent <= E2))
    acc = float(accuracy_score(y_true[kn], y_prob_cal.argmax(axis=1)[kn]))
    print("")
    print("  retention floor {:.0%}  ->  margin >= {:.2f}   entropy <= {:.2f}".format(
        floor, M2, E2))
    print("    retention  nominal {:.1%} ({} of {})   worst {:.1%}".format(
        float(kn.mean()), int(kn.sum()), len(y_true), RW))
    print("    rejection  nominal {:.1%} ({} of {})   worst {:.1%}".format(
        float(1 - kr.mean()), round((1 - kr.mean()) * len(or_ent)),
        len(or_ent), RJW))
    print("    accuracy on accepted {:.2%}".format(acc))
    break

# ---- and the honest range at the pair cell 4 chose --------------------------
print("")
print("=" * 70)
print("E. CELL 4's CHOICE, STRESSED WITH THE MEASURED BAND")
print("=" * 70)
candidates = [(MARGIN_THRESHOLD, ENTROPY_THRESHOLD, "cell 4 as fitted")]
if band_best is not None and (abs(band_best[0] - MARGIN_THRESHOLD) > 1e-9 or
                              abs(band_best[1] - ENTROPY_THRESHOLD) > 1e-9):
    candidates.append((round(band_best[0], 2), round(band_best[1], 2),
                       "band-aware refit"))

for M2, E2, tag in candidates:
    kn = ((id_top1 >= CONFIDENCE_THRESHOLD) & (id_marg >= M2) & (id_ent <= E2))
    kr = ((or_top1 >= CONFIDENCE_THRESHOLD) & (or_marg >= M2) & (or_ent <= E2))
    acc = float(accuracy_score(y_true[kn], y_prob_cal.argmax(axis=1)[kn]))
    print("  {:<20} margin >= {:.2f}  entropy <= {:.2f}".format(tag, M2, E2))
    print("    retention  nominal {:.1%}   worst {:.1%}".format(
        float(kn.mean()), worst_ret(M2, E2)))
    print("    rejection  nominal {:.1%} ({} of {})   worst {:.1%}".format(
        float(1 - kr.mean()), round((1 - kr.mean()) * len(or_ent)),
        len(or_ent), worst_rej(M2, E2, or_top1, or_marg, or_ent)))
    print("    accuracy on accepted {:.2%}".format(acc))
    print("    test images within {:.4f} of the margin line  : {} of {}".format(
        BAND_M, int((np.abs(id_marg - M2) <= BAND_M).sum()), len(id_marg)))
    print("    test images within {:.4f} of the entropy line : {} of {}".format(
        BAND_E, int((np.abs(id_ent - E2) <= BAND_E).sum()), len(id_ent)))
    print("")

In [ ]:
# ============================================================================
# CELL 7 - Scale the simulated bands up to the real export measurement, then
# re-stress the chosen pair. No prediction runs; arithmetic only.
# ============================================================================
REAL_CONF_SHIFT = FLOAT16_SHIFT          # 0.042463, measured on the real export
sim_max = float(d_conf.max())
SCALE = REAL_CONF_SHIFT / sim_max if sim_max > 0 else 1.0

print("The simulation in cell 6 measures weight quantization only. The phone")
print("also runs WEBGL_FORCE_F16_TEXTURES, so activations are 16-bit as well.")
print("  simulated confidence max : {:.6f}".format(sim_max))
print("  real export, cell #32    : {:.6f}".format(REAL_CONF_SHIFT))
print("  scale factor             : {:.3f}x".format(SCALE))
if SCALE < 1.0:
    print("")
    print("  Scale below 1.0: the simulation already moved confidence further")
    print("  than the real export did. Bands are used unscaled, which is the")
    print("  more conservative of the two.")
    SCALE = 1.0
print("")

# The pair cell 4 chose, plus a looser fallback in case the stress test kills it.
fallback = (round(max(0.01, MARGIN_THRESHOLD - 0.06), 2),
            round(min(0.99, ENTROPY_THRESHOLD + 0.02), 2))
pairs = [(MARGIN_THRESHOLD, ENTROPY_THRESHOLD)]
if fallback != (MARGIN_THRESHOLD, ENTROPY_THRESHOLD):
    pairs.append(fallback)

for tag, use_max in [("worst case  (max shift, scaled)", True),
                     ("realistic   (p99 shift, scaled)", False)]:
    BE = (float(d_ent.max())  if use_max else float(np.percentile(d_ent, 99))) * SCALE
    BM = (float(d_marg.max()) if use_max else float(np.percentile(d_marg, 99))) * SCALE
    print("=" * 70)
    print("{}   entropy band {:.6f}   margin band {:.6f}".format(tag, BE, BM))
    print("=" * 70)
    for M2, E2 in pairs:
        kw = ((id_top1 >= CONFIDENCE_THRESHOLD) &
              (id_marg >= M2 + BM) & (id_ent <= E2 - BE))
        kj = ((or_top1 >= CONFIDENCE_THRESHOLD) &
              (or_marg >= max(0.0, M2 - BM)) & (or_ent <= E2 + BE))
        flag = "" if float(kw.mean()) >= MIN_ID_RETENTION else "   <- below floor"
        print("  margin >= {:.2f}  entropy <= {:.2f}   retention {:.1%}"
              "   rejection {:.1%} ({} of {}){}".format(
                  M2, E2, float(kw.mean()), float(1 - kj.mean()),
                  round((1 - kj.mean()) * len(or_ent)), len(or_ent), flag))
    print("")

print("=" * 70)
print("VERDICT")
print("=" * 70)
BE_w = float(d_ent.max())  * SCALE
BM_w = float(d_marg.max()) * SCALE
kw = ((id_top1 >= CONFIDENCE_THRESHOLD) &
      (id_marg >= MARGIN_THRESHOLD + BM_w) & (id_ent <= ENTROPY_THRESHOLD - BE_w))
ret_w = float(kw.mean())
if ret_w >= MIN_ID_RETENTION:
    print("  margin {:.2f} / entropy {:.2f} holds {:.1%} retention even in the".format(
        MARGIN_THRESHOLD, ENTROPY_THRESHOLD, ret_w))
    print("  worst direction of the scaled band. Safe to deploy.")
else:
    print("  margin {:.2f} / entropy {:.2f} drops to {:.1%} retention in the worst".format(
        MARGIN_THRESHOLD, ENTROPY_THRESHOLD, ret_w))
    print("  direction, below the {:.0%} floor. Two options, in order of".format(
        MIN_ID_RETENTION))
    print("  preference:")
    print("    1. loosen entropy toward the fallback pair printed above")
    print("    2. accept the lower worst-case retention and state it in Limitations")
    print("  Do not loosen the confidence threshold. It stays at 0.70.")

## Cell 8 — The new `model_config.json`

This is the only file that changes in the app. The weight shards, `model.json`, and
`src/lib/classifier.ts` all stay exactly as they are. `classifier.ts` already reads
`marginThreshold` and `entropyThreshold` out of the config and treats `null` as "check
disabled", so filling in two numbers turns both checks on with no code edit.

### Version bumps that must follow

Installed phones keep serving the old config out of the service-worker cache forever unless the
cache name changes:

| File | From | To |
| :-- | :-- | :-- |
| `package.json` | `0.2.7` | `0.3.0` |
| `src/lib/constants.ts` | `model-v2.1.1` | `model-v3.0.0` |
| `src/sw.ts` — `MODEL_CACHE` | `mangoleaf-model-cache-v2-1-1` | `mangoleaf-model-cache-v3-0-0` |
| `src/sw.ts` — `OUTDATED_MODEL_CACHES` | — | add `"mangoleaf-model-cache-v2-1-1"` |

Then `bun run build`, and check the generated `public/sw.js` precache manifest actually lists
the new model files before deploying.

### Before deploying, confirm on the phone

Open the console on `https://mangoleaf-ai.vercel.app` and look for
`Model config loaded: margin=..., entropy=...`. Neither value may be `null`. Then photograph
something that is not a mango and confirm it comes back as Uncertain rather than a disease.

### The guard

This cell refuses to write anything if the margin lands below 0.05. A margin that small sits
inside the float16 band measured in cell 6, which means the phone could accept a prediction
this notebook rejected. If the guard fires, do not work around it — the margin has to be raised
by hand and cell 7 re-read.

In [ ]:
# ============================================================================
# CELL 8 - Write the new model_config.json. Nothing is trained or overwritten.
# ============================================================================
import shutil

if reject_cfg is None:
    raise RuntimeError("Cell 4 did not produce thresholds. Nothing to write.")

MARGIN_FLOOR = 0.05
if reject_cfg["marginThreshold"] < MARGIN_FLOOR:
    raise RuntimeError(
        "REFUSING TO WRITE.\n"
        "margin {:.4f} is below the {:.2f} float16 safety band. The export shifts "
        "confidence by up to {:.6f}, so the phone could accept a prediction this "
        "notebook rejected. Raise the margin by hand, re-read cell 7, then rerun "
        "this cell.".format(reject_cfg["marginThreshold"], MARGIN_FLOOR,
                            FLOAT16_SHIFT))

new_model_version = "model-v3.0.0"

# Worst-case figures from the scaled band in cell 7, recorded for the manuscript.
kw = ((id_top1 >= CONFIDENCE_THRESHOLD) &
      (id_marg >= MARGIN_THRESHOLD + BM_w) & (id_ent <= ENTROPY_THRESHOLD - BE_w))
kj = ((or_top1 >= CONFIDENCE_THRESHOLD) &
      (or_marg >= max(0.0, MARGIN_THRESHOLD - BM_w)) &
      (or_ent <= ENTROPY_THRESHOLD + BE_w))
f16_worst_retention = round(float(kw.mean()), 4)
f16_worst_rejection = round(float(1.0 - kj.mean()), 4)

rejection_block = dict(reject_cfg)
rejection_block["float16WorstCaseRetention"] = f16_worst_retention
rejection_block["float16WorstCaseRejection"] = f16_worst_rejection

model_config = {
    "modelVersion": new_model_version,
    "trial": EXPECTED_TRIAL,
    "classes": CLASS_NAMES,
    "inputSize": [IMG_SIZE[0], IMG_SIZE[1]],
    "pixelRange": "0-255 raw, MobileNetV3 rescales internally",
    "temperature": round(T, 6),
    "temperatureFoldedIntoWeights": True,
    "thresholds": {
        "confidenceThreshold": CONFIDENCE_THRESHOLD,
        "marginThreshold": reject_cfg["marginThreshold"],
        "entropyThreshold": reject_cfg["entropyThreshold"],
        "note": ("margin and entropy fitted on {} out-of-scope images; rejection "
                 "rate {:.1%} measured on {} held-out images".format(
                     reject_cfg["oodTuneCount"], reject_cfg["reportRejection"],
                     reject_cfg["oodReportCount"])),
    },
    "rejection": rejection_block,
}

cfg_path = os.path.join(DRIVE_SAVE, "model_config_v3.json")
with open(cfg_path, "w") as f:
    json.dump(model_config, f, indent=2)
print("Wrote {}".format(cfg_path))

# The refit histogram goes next to it under its own name. The training run's
# rejection_distributions.png is cited by the manuscript and is left untouched.
fig_dst_dir = os.path.join(DRIVE_SAVE, "figures")
os.makedirs(fig_dst_dir, exist_ok=True)
src_fig = os.path.join(FIG_DIR, "rejection_distributions_refit.png")
if os.path.exists(src_fig):
    shutil.copy(src_fig, os.path.join(fig_dst_dir,
                                      "rejection_distributions_refit.png"))
    print("Copied rejection_distributions_refit.png to {}".format(fig_dst_dir))

print("")
print("=" * 70)
print("PASTE THIS INTO public/model/model_config.json")
print("=" * 70)
print(json.dumps(model_config, indent=2))

print("")
print("=" * 70)
print("NUMBERS FOR THE MANUSCRIPT")
print("=" * 70)
print("  1. confidence threshold      : {:.2f}   (fixed, cited to [31])".format(
    CONFIDENCE_THRESHOLD))
print("  2. margin threshold          : {:.2f}   (selected among ties, not fitted)".format(
    reject_cfg["marginThreshold"]))
print("  3. entropy threshold         : {:.2f}".format(reject_cfg["entropyThreshold"]))
print("  4. in-distribution retention : {:.1%}  ({} of {} test images kept)".format(
    reject_cfg["tuneRetention"],
    int(((id_top1 >= CONFIDENCE_THRESHOLD) &
         (id_marg >= MARGIN_THRESHOLD) & (id_ent <= ENTROPY_THRESHOLD)).sum()),
    len(y_true)))
print("  5. out-of-scope rejection    : {:.1%}  ({} of {} held-out images)".format(
    reject_cfg["reportRejection"],
    round(reject_cfg["reportRejection"] * reject_cfg["oodReportCount"]),
    reject_cfg["oodReportCount"]))
se = (reject_cfg["reportRejection"] * (1 - reject_cfg["reportRejection"]) /
      reject_cfg["oodReportCount"]) ** 0.5
print("     95% CI                    : {:.1%} to {:.1%}".format(
    max(0.0, reject_cfg["reportRejection"] - 1.96 * se),
    min(1.0, reject_cfg["reportRejection"] + 1.96 * se)))
print("  6. accuracy on accepted      : {:.2%}".format(reject_cfg["accuracyOnAccepted"]))
print("  7. out-of-scope images used  : {} tune + {} report = {}".format(
    reject_cfg["oodTuneCount"], reject_cfg["oodReportCount"],
    reject_cfg["oodTuneCount"] + reject_cfg["oodReportCount"]))
print("")
print("  Say the margin was SELECTED AMONG TIES, not fitted. Several margins")
print("  gave the same rejection rate; the largest was taken because it costs")
print("  nothing and makes the gate do real work. Cell 5 printed the tie set.")
print("")
print("  Test accuracy is unchanged at {:.2%}. This notebook did not train.".format(
    float(resume_info["testAccuracy"])))